In [ ]:
!pip install pytorch_fid
!pip install torchmetrics[image] torch-fidelity

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import math
from pytorch_fid.fid_score import calculate_fid_given_paths
from torchmetrics.image.inception import InceptionScore
import torch.utils.data as data
import os
from PIL import Image

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor()
])

train_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=False,
    transform=transform
)
train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)

In [ ]:
T = 1000
beta_start = 1e-4
beta_end = 0.02
betas = torch.linspace(
    beta_start,
    beta_end,
    T,
    device=device
)
alphas = 1.0 - betas
alpha_time_t = torch.cumprod(alphas, dim=0)

In [ ]:
def q_sample(x0, t, noise=None):
    if noise is None:
        noise = torch.randn_like(x0)

    sqrt_alpha_time_t = torch.sqrt(alpha_time_t[t])
    sqrt_one_minus_alpha_time_t = torch.sqrt(1 - alpha_time_t[t])

    # Reshape so the values can be multiplied across the entire picture
    sqrt_alpha_time_t = sqrt_alpha_time_t[:, None, None, None]
    sqrt_one_minus_alpha_time_t = sqrt_one_minus_alpha_time_t[:, None, None, None]

    xt = (
        sqrt_alpha_time_t * x0
        + sqrt_one_minus_alpha_time_t * noise
    )

    return xt, noise

In [ ]:
class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        device = t.device

        half_dim = self.dim // 2

        emb_scale = math.log(10000) / (half_dim - 1)

        emb = torch.exp(
            torch.arange(half_dim, device=device) * -emb_scale
        )

        emb = t.float()[:, None] * emb[None, :]

        emb = torch.cat(
            [torch.sin(emb), torch.cos(emb)],
            dim=1
        )

        return emb

In [ ]:
class block(nn.Module):
    def __init__(self, in_channels, out_channels, num_groups=8):
        super().__init__()
        self.normalize = nn.GroupNorm(num_groups, in_channels)
        self.convolute = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.act_func = nn.SiLU()

    def forward(self, x):
        x = self.normalize(x)
        x = self.act_func(x)
        x = self.convolute(x)
        return x


In [ ]:
class resnetblock(nn.Module):
    def __init__(self, in_channels, out_channels, time_dim):
        super().__init__()
        self.mlp = nn.Linear(time_dim, out_channels)
        self.block1 = block(in_channels, out_channels)
        self.block2 = block(out_channels, out_channels)

        if in_channels != out_channels:
            self.residual = nn.Conv2d(in_channels, out_channels, kernel_size = 1)
        else: self.residual = nn.Identity()

    def forward(self, x, time_emb):
        h = self.block1(x)
        time_emb = self.mlp(time_emb)
        time_emb = time_emb[:, :, None, None]
        h = h + time_emb
        h = self.block2(h)

        return h + self.residual(x)



In [ ]:
class UNet(nn.Module):
    def __init__(self, in_channels=3, base_channels=64, time_dim=128):
        super().__init__()
        self.time_embedding = nn.Sequential(
            SinusoidalTimeEmbedding(time_dim),
            nn.Linear(time_dim, time_dim),
            nn.SiLU(),
            nn.Linear(time_dim, time_dim)
        )
        #initial convolution to get 64 channels from the base 1 channel since the pictures are greyscale
        self.input_conv = nn.Conv2d(in_channels, base_channels, kernel_size=3, padding=1)

        #first downblock, convoluting and learning features while sending skip connection to upblocks
        self.downblock1 = resnetblock(base_channels, base_channels, time_dim)

        #first downsample, where number of channels increases for more feature learning and stride=2 to turn the 28 by 28 pictures into 14 by 14
        self.downsample1 = nn.Conv2d(base_channels, base_channels * 2, kernel_size = 4, stride=2, padding=1)

        self.downblock2 = resnetblock(base_channels * 2, base_channels * 2, time_dim)

        self.downsample2 = nn.Conv2d(base_channels * 2, base_channels * 4, kernel_size=4, stride=2, padding=1)

        self.bottleneck1 = resnetblock(base_channels * 4, base_channels * 4, time_dim)

        self.bottleneck2 = resnetblock(base_channels * 4, base_channels * 4, time_dim)

        #first upsample, reducing number of channels and increasing from 7 by 7 image to 14 by 14 using convtranspose2d
        self.upsample1 = nn.ConvTranspose2d(base_channels * 4, base_channels * 2, kernel_size = 4, stride=2, padding=1)

        # Fix: input channels for upblock1 should be base_channels * 4 due to concatenation
        self.upblock1 = resnetblock(base_channels * 4, base_channels * 2, time_dim)

        self.upsample2 = nn.ConvTranspose2d(base_channels * 2, base_channels, kernel_size=4, stride=2, padding=1)

        # Fix: input channels for upblock2 should be base_channels * 2 due to concatenation
        self.upblock2 = resnetblock(base_channels * 2, base_channels, time_dim)

        self.output = nn.Conv2d(base_channels, in_channels, kernel_size=1)

    def forward(self, x, t):
        t_emb = self.time_embedding(t)
        x = self.input_conv(x)
        d1 = self.downblock1(x, t_emb)
        x = self.downsample1(d1)
        d2 = self.downblock2(x, t_emb)
        x = self.downsample2(d2)
        x = self.bottleneck1(x, t_emb)
        x = self.bottleneck2(x, t_emb)
        x = self.upsample1(x)
        x = torch.cat([x, d2], dim=1)
        u1 = self.upblock1(x, t_emb)
        x = self.upsample2(u1)
        x = torch.cat([x, d1], dim=1)
        u2 = self.upblock2(x, t_emb)
        return self.output(u2)


In [ ]:
model = UNet().to(device)

Starting with Linear Schedule DDPM on CIFAR-10 (no MNIST since that was just for practicing building the model):

In [ ]:
PATH = "/checkpoints/ddpm_cifar_linear.pth"
checkpoint = torch.load(PATH)
model.load_state_dict(checkpoint['model_state_dict'])

In [ ]:
@torch.no_grad()
def sample_ddpm(
    model,
    n_samples,
    image_size=32
):
    model.eval()

    x = torch.randn(
        n_samples,
        3, #in channels value
        image_size,
        image_size,
        device=device
    )

    for t_value in reversed(range(T)):

        t = torch.full(
            (n_samples,),
            t_value,
            device=device,
            dtype=torch.long
        )

        # Predict noise
        pred_noise = model(x, t)

        beta_t = betas[t_value]
        alpha_t = alphas[t_value]
        alpha_bar_t = alpha_time_t[t_value]

        # DDPM mean
        mean = (
            1 / torch.sqrt(alpha_t)
        ) * (
            x
            -
            (beta_t / torch.sqrt(1 - alpha_bar_t))
            * pred_noise
        )

        if t_value > 0:

            noise = torch.randn_like(x)

            variance = beta_t

            x = (
                mean
                + torch.sqrt(variance) * noise
            )

        else:
            x = mean

    return x

In [ ]:
generated_dir = "./generated_cifar"
os.makedirs(generated_dir, exist_ok=True)

def batch_sample_gen(n_samples, batches, model, generated_dir):

    inception_score = InceptionScore(
        splits=12,
        normalize=True
    )

    os.makedirs(generated_dir, exist_ok=True)

    samples_per_batch = n_samples // batches

    for i in range(batches):

        print("Batch number:", i)

        samples = sample_ddpm(
            model,
            n_samples=samples_per_batch
        )

        # Convert [-1, 1] to [0, 1]
        samples = (samples.clamp(-1, 1) + 1) / 2

        # Update IS using normalized images
        is_samples = samples[0:100]
        inception_score.update(is_samples.to('cpu'))

        # Convert to uint8 for saving
        images = (samples * 255).round().to(torch.uint8)

        for j, img in enumerate(images):

            # [C, H, W] -> [H, W, C]
            img = img.permute(1, 2, 0).cpu().numpy()

            pic_no = i * samples_per_batch + j

            Image.fromarray(img).save(
                os.path.join(
                    generated_dir,
                    f"{pic_no:05d}.png"
                )
            )

    # Calculate IS once across all generated images
    is_mean, is_std = inception_score.compute()

    return is_mean.item(), is_std.item()


In [ ]:
is_mean, is_std = batch_sample_gen(n_samples=3000, batches=30, model=model, generated_dir=generated_dir)

In [ ]:
real_dir = "./real_cifar"
os.makedirs(real_dir, exist_ok=True)

counter = 0
n_samples = 3000

for images, _ in train_loader:
    for image in images:
        if counter >= n_samples:
            break

        # Convert [-1, 1] -> [0, 1] -> [0, 255]
        image = (image.clamp(-1, 1) + 1) / 2
        image = (image * 255).round().to(torch.uint8)

        # [C, H, W] -> [H, W, C]
        image = image.permute(1, 2, 0).cpu().numpy()

        Image.fromarray(image).save(
            os.path.join(real_dir, f"{counter:05d}.png")
        )

        counter += 1

    if counter >= n_samples:
        break

In [ ]:
fid_value = calculate_fid_given_paths(
    paths=['./real_cifar', './generated_cifar'],
    batch_size=100,
    device='cuda',
    dims=2048
)
print(f"FID score: {fid_value}")
print(f"IS score and stddev: {is_mean}, {is_std}")

Now using DDPM with a Cosine Schedule:

In [ ]:
PATH = "/checkpoints/ddpm_cifar_cosine.pth"
checkpoint = torch.load(PATH)
model.load_state_dict(checkpoint['model_state_dict'])

In [ ]:
T = 1000
s = 8e-3

# Standard cosine beta schedule (corrected step division)
steps = torch.linspace(
    0, T, T + 1,
    dtype=torch.float64,
    device=device
)
alpha_bar = torch.cos(((steps / T + s) / (1 + s)) * math.pi / 2) ** 2
alpha_bar = alpha_bar / alpha_bar[0]

betas = 1 - alpha_bar[1:] / alpha_bar[:-1]
betas = betas.clamp(min=1e-5, max=0.999).float()

alphas = 1.0 - betas
alpha_time_t = torch.cumprod(alphas, dim=0)

In [ ]:
is_mean, is_std = batch_sample_gen(n_samples=3000, batches=30, model=model, generated_dir=generated_dir)

In [ ]:
fid_value = calculate_fid_given_paths(
    paths=['./real_cifar', './generated_cifar'],
    batch_size=100,
    device='cuda',
    dims=2048
)
print(f"FID score: {fid_value}")
print(f"IS score and stddev: {is_mean}, {is_std}")

Now moving on to DDIM sampling at 1000, 50 and 10 steps:

In [ ]:
@torch.no_grad()
def sample_ddim(
    model,
    n_samples,
    steps,
    eta=1.0,
    image_size=32,  
):
    model.eval()

    x = torch.randn(
        n_samples,
        3,
        image_size,
        image_size,
        device=device
    )

    bigT = torch.linspace(
        0, T-1, steps=steps, device=device
    ).round().long().flip(0)

    for i, t_value in enumerate(bigT):
        t_value = int(t_value.item())

        if i + 1 < len(bigT):
            prev_t = int(bigT[i + 1].item())
            alpha_bar_prev = alpha_time_t[prev_t]
        else:
            alpha_bar_prev = torch.ones(
                (), device=device, dtype=alpha_time_t.dtype
            )

        alpha_bar_t = alpha_time_t[t_value]

        t = torch.full(
            (n_samples,),
            t_value,
            device=device,
            dtype=torch.long
        )

        # Predict noise
        pred_noise = model(x, t)

        # Predict original image
        x0_pred = (
            x - torch.sqrt(1 - alpha_bar_t) * pred_noise
        ) / torch.sqrt(alpha_bar_t)

        # Calculate DDIM variance (sigma)
        sigma = eta * torch.sqrt(
            ((1 - alpha_bar_prev) / (1 - alpha_bar_t))
            * (1 - alpha_bar_t / alpha_bar_prev)
        )

        # Calculate direction pointing toward noise
        direction = torch.sqrt(
            (1 - alpha_bar_prev - sigma**2).clamp(min=0)
        ) * pred_noise

        # Add stochastic noise
        noise = torch.randn_like(x) if eta > 0 else torch.zeros_like(x)

        # Update sample
        x = (
            torch.sqrt(alpha_bar_prev) * x0_pred
            + direction
            + sigma * noise
        )

    return x


In [ ]:

def batch_sample_gen(n_samples, batches, model, generated_dir, steps, eta):

    inception_score = InceptionScore(
        splits=12,
        normalize=True
    )

    os.makedirs(generated_dir, exist_ok=True)

    samples_per_batch = n_samples // batches

    for i in range(batches):

        print("Batch number:", i)

        samples = sample_ddim(
            model,
            n_samples=samples_per_batch,
            steps=steps,
            eta=eta
        )

        # Convert [-1, 1] to [0, 1]
        samples = (samples.clamp(-1, 1) + 1) / 2

        # Update IS using normalized images
        is_samples = samples[0:100]
        inception_score.update(is_samples.to('cpu'))

        # Convert to uint8 for saving
        images = (samples * 255).round().to(torch.uint8)

        for j, img in enumerate(images):

            # [C, H, W] -> [H, W, C]
            img = img.permute(1, 2, 0).cpu().numpy()

            pic_no = i * samples_per_batch + j

            Image.fromarray(img).save(
                os.path.join(
                    generated_dir,
                    f"{pic_no:05d}.png"
                )
            )

    # Calculate IS once across all generated images
    is_mean, is_std = inception_score.compute()

    return is_mean.item(), is_std.item()


In [ ]:
batch_sample_gen(3000, 30, model=model, generated_dir=generated_dir, steps=1000, eta=1.0)

In [ ]:
fid_value = calculate_fid_given_paths(
    paths=['./real_cifar', './generated_cifar'],
    batch_size=100,
    device='cuda',
    dims=2048
)
print(f"FID score: {fid_value}")
print(f"IS score and stddev: {is_mean}, {is_std}")

In [ ]:
batch_sample_gen(3000, 30, model=model, generated_dir=generated_dir, steps=100, eta=0.2)

In [ ]:
fid_value = calculate_fid_given_paths(
    paths=['./real_cifar', './generated_cifar'],
    batch_size=100,
    device='cuda',
    dims=2048
)
print(f"FID score: {fid_value}")
print(f"IS score and stddev: {is_mean}, {is_std}")

In [ ]:
batch_sample_gen(3000, 30, model=model, generated_dir=generated_dir, steps=50, eta=0.1)

In [ ]:
fid_value = calculate_fid_given_paths(
    paths=['./real_cifar', './generated_cifar'],
    batch_size=100,
    device='cuda',
    dims=2048
)
print(f"FID score: {fid_value}")
print(f"IS score and stddev: {is_mean}, {is_std}")

Now setting Classifier Free Guidance:

In [ ]:
class UNet(nn.Module):
    def __init__(self, in_channels=3, base_channels=64, time_dim=128):
        super().__init__()
        self.time_embedding = nn.Sequential(
            SinusoidalTimeEmbedding(time_dim),
            nn.Linear(time_dim, time_dim),
            nn.SiLU(),
            nn.Linear(time_dim, time_dim)
        )
        self.class_embedding = nn.Embedding(11, time_dim)
        #initial convolution to get 64 channels from the base 1 channel since the pictures are greyscale
        self.input_conv = nn.Conv2d(in_channels, base_channels, kernel_size=3, padding=1)

        #first downblock, convoluting and learning features while sending skip connection to upblocks
        self.downblock1 = resnetblock(base_channels, base_channels, time_dim)

        #first downsample, where number of channels increases for more feature learning and stride=2 to turn the 28 by 28 pictures into 14 by 14
        self.downsample1 = nn.Conv2d(base_channels, base_channels * 2, kernel_size = 4, stride=2, padding=1)

        self.downblock2 = resnetblock(base_channels * 2, base_channels * 2, time_dim)

        self.downsample2 = nn.Conv2d(base_channels * 2, base_channels * 4, kernel_size=4, stride=2, padding=1)

        self.bottleneck1 = resnetblock(base_channels * 4, base_channels * 4, time_dim)

        self.bottleneck2 = resnetblock(base_channels * 4, base_channels * 4, time_dim)

        #first upsample, reducing number of channels and increasing from 7 by 7 image to 14 by 14 using convtranspose2d
        self.upsample1 = nn.ConvTranspose2d(base_channels * 4, base_channels * 2, kernel_size = 4, stride=2, padding=1)

        # Fix: input channels for upblock1 should be base_channels * 4 due to concatenation
        self.upblock1 = resnetblock(base_channels * 4, base_channels * 2, time_dim)

        self.upsample2 = nn.ConvTranspose2d(base_channels * 2, base_channels, kernel_size=4, stride=2, padding=1)

        # Fix: input channels for upblock2 should be base_channels * 2 due to concatenation
        self.upblock2 = resnetblock(base_channels * 2, base_channels, time_dim)

        self.output = nn.Conv2d(base_channels, in_channels, kernel_size=1)

    def forward(self, x, t, labels):
        class_emb = self.class_embedding(labels)
        t_emb = self.time_embedding(t)
        t_emb = t_emb + class_emb
        x = self.input_conv(x)
        d1 = self.downblock1(x, t_emb)
        x = self.downsample1(d1)
        d2 = self.downblock2(x, t_emb)
        x = self.downsample2(d2)
        x = self.bottleneck1(x, t_emb)
        x = self.bottleneck2(x, t_emb)
        x = self.upsample1(x)
        x = torch.cat([x, d2], dim=1)
        u1 = self.upblock1(x, t_emb)
        x = self.upsample2(u1)
        x = torch.cat([x, d1], dim=1)
        u2 = self.upblock2(x, t_emb)
        return self.output(u2)

In [ ]:
model = UNet().to(device)
PATH = "/checkpoints/cfg_cifar.pth"
checkpoint = torch.load(PATH)
model.load_state_dict(checkpoint['model_state_dict'])

In [ ]:
def guided_noise(model, x, t, labels, guidance_scale):

    uncond_labels = torch.full_like(labels, 10)

    noise_uncond = model(x, t, uncond_labels)
    noise_cond = model(x, t, labels)

    guided_noise = noise_uncond + guidance_scale * (
        noise_cond - noise_uncond
    )

    return guided_noise

In [ ]:
@torch.no_grad()
def sample_cfg_ddim(
    model,
    n_samples,
    label_to_gen, #which images to generate, e.g. 5 = dogs (from 0 to 10)
    image_size=32,
    steps=1000,
    eta=1.0,
    guidance_scale=1.0 #setting how much the guidance influences the generation
):
    model.eval()

    labels = torch.full((n_samples,), label_to_gen, dtype=torch.long, device=device)

    x = torch.randn(
        n_samples,
        3,
        image_size,
        image_size,
        device=device
    )

    bigT = torch.linspace(
        0, T-1, steps=steps, device=device
    ).round().long().flip(0)

    for i, t_value in enumerate(bigT):
        t_value = int(t_value.item())

        if i + 1 < len(bigT):
            prev_t = int(bigT[i + 1].item())
            alpha_bar_prev = alpha_time_t[prev_t]
        else:
            alpha_bar_prev = torch.ones(
                (), device=device, dtype=alpha_time_t.dtype
            )

        alpha_bar_t = alpha_time_t[t_value]

        t = torch.full(
            (n_samples,),
            t_value,
            device=device,
            dtype=torch.long
        )

        # Predict noise
        pred_noise = guided_noise(model, x, t, labels, guidance_scale)


        # Predict original image
        x0_pred = (
            x - torch.sqrt(1 - alpha_bar_t) * pred_noise
        ) / torch.sqrt(alpha_bar_t)

        # Calculate DDIM variance (sigma)
        sigma = eta * torch.sqrt(
            ((1 - alpha_bar_prev) / (1 - alpha_bar_t))
            * (1 - alpha_bar_t / alpha_bar_prev)
        )

        # Calculate direction pointing toward noise
        direction = torch.sqrt(
            (1 - alpha_bar_prev - sigma**2).clamp(min=0)
        ) * pred_noise

        # Add stochastic noise
        noise = torch.randn_like(x) if eta > 0 else torch.zeros_like(x)

        # Update sample
        x = (
            torch.sqrt(alpha_bar_prev) * x0_pred
            + direction
            + sigma * noise
        )

    return x


In [ ]:

def batch_sample_gen(n_samples, batches, model, generated_dir, label_to_gen):

    inception_score = InceptionScore(
        splits=12,
        normalize=True
    )

    os.makedirs(generated_dir, exist_ok=True)

    samples_per_batch = n_samples // batches

    for i in range(batches):

        print("Batch number:", i)

        samples = sample_ddim(
            model,
            n_samples=samples_per_batch, label_to_gen=label_to_gen
        )

        # Convert [-1, 1] to [0, 1]
        samples = (samples.clamp(-1, 1) + 1) / 2

        # Update IS using normalized images
        is_samples = samples[0:100]
        inception_score.update(is_samples.to('cpu'))

        # Convert to uint8 for saving
        images = (samples * 255).round().to(torch.uint8)

        for j, img in enumerate(images):

            # [C, H, W] -> [H, W, C]
            img = img.permute(1, 2, 0).cpu().numpy()

            pic_no = i * samples_per_batch + j

            Image.fromarray(img).save(
                os.path.join(
                    generated_dir,
                    f"{pic_no:05d}.png"
                )
            )

    # Calculate IS once across all generated images
    is_mean, is_std = inception_score.compute()

    return is_mean.item(), is_std.item()


In [ ]:
def gen_real_comparisons(target_label):    
    real_dir = "./real_cifar"
    os.makedirs(real_dir, exist_ok=True)

    counter = 0
    n_samples = 3000

    for images, labels in train_loader:
        mask = labels == target_label
        images = images[mask]
        for image in images:
            if counter >= n_samples:
                break
            # Convert [-1, 1] -> [0, 1] -> [0, 255]
            image = (image.clamp(-1, 1) + 1) / 2
            image = (image * 255).round().to(torch.uint8)

            # [C, H, W] -> [H, W, C]
            image = image.permute(1, 2, 0).cpu().numpy()

            Image.fromarray(image).save(
                os.path.join(real_dir, f"{counter:05d}.png")
            )

            counter += 1

        if counter >= n_samples:
            break

In [ ]:
for target_label in range(0, 10):
    if target_label == 10:
        is_mean, is_std = batch_sample_gen(n_samples=3000,batches=30, model=model, generated_dir=generated_dir, label_to_gen=target_label)
        real_dir = "./real_cifar"
        os.makedirs(real_dir, exist_ok=True)
        counter = 0
        n_samples = 3000
        for images, labels in train_loader:
            for image in images:
                if counter >= n_samples:
                    break
                # Convert [-1, 1] -> [0, 1] -> [0, 255]
                image = (image.clamp(-1, 1) + 1) / 2
                image = (image * 255).round().to(torch.uint8)

                # [C, H, W] -> [H, W, C]
                image = image.permute(1, 2, 0).cpu().numpy()

                Image.fromarray(image).save(
                    os.path.join(real_dir, f"{counter:05d}.png")
                )

                counter += 1

            if counter >= n_samples:
                break
    else:
        is_mean, is_std = batch_sample_gen(n_samples=3000,batches=30, model=model, generated_dir=generated_dir, label_to_gen=target_label)
        gen_real_comparisons(target_label)
    fid_value = calculate_fid_given_paths(
    paths=['./real_cifar', './generated_cifar'],
    batch_size=100,
    device='cuda',
    dims=2048
    )
    print(f"FID score: {fid_value}")
    print(f"IS score and stddev: {is_mean}, {is_std}")

Testing the Consistency Model:

In [ ]:
import hashlib
import json
import math
import tempfile
import time
from datetime import datetime, timezone
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from PIL import Image
from pytorch_fid.fid_score import calculate_fid_given_paths

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    raise RuntimeError("Select a GPU runtime in Colab before running this notebook.")
print("Device:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)

In [ ]:
TEACHER_PATH = Path("/checkpoints/ddpm_cifar_250.pth")

In [ ]:
class ConsistencyStudent(nn.Module):
    def __init__(self, alpha_cumprod, sigma_data=0.5):
        super().__init__()
        self.unet = UNet()
        self.register_buffer('noise_std', torch.sqrt(1 - alpha_cumprod.clone()))
        self.sigma_data = sigma_data

    def forward(self, x, t):
        if x.ndim != 4 or x.shape[1:] != (3, 32, 32):
            raise ValueError('x must have shape [batch, 3, 32, 32].')
        if t.dtype != torch.long or t.ndim != 1 or t.shape[0] != x.shape[0]:
            raise ValueError('t must be a torch.long vector with one index per image.')
        if x.device != self.noise_std.device or t.device != x.device:
            raise ValueError('x, t, and the student must be on the same device.')
        if torch.any(t < 0) or torch.any(t >= self.noise_std.numel()):
            raise ValueError('timestep indices must be between 0 and 999.')
        sigma = self.noise_std[t].view(-1, 1, 1, 1)
        offset = sigma - self.noise_std[0]
        c_skip = self.sigma_data ** 2 / (offset ** 2 + self.sigma_data ** 2)
        c_out = self.sigma_data * offset / torch.sqrt(sigma ** 2 + self.sigma_data ** 2)
        return c_skip * x + c_out * self.unet(x, t)

In [ ]:
teacher = UNet().to(device)
teacher_weights = torch.load(TEACHER_PATH, map_location=device, weights_only=True)
print(teacher.load_state_dict(teacher_weights))
del teacher_weights
teacher.eval()
teacher.requires_grad_(False)
print('Teacher parameters:', sum(p.numel() for p in teacher.parameters()))

student = ConsistencyStudent(alpha_time_t).to(device)
student.unet.load_state_dict(teacher.state_dict())
student.train()
optimizer = torch.optim.Adam(student.parameters(), lr=1e-4)
batch_rng = torch.Generator(device="cpu").manual_seed(56789)
print("Student U-Net initialized from the teacher.")

In [ ]:
levels = torch.linspace(0, 998, steps=101).round().long().tolist()
assert levels[-3:] == [978, 988, 998]
assert all(a < b for a, b in zip(levels, levels[1:]))
substep_path = list(range(998, 987, -1)) + levels[-3::-1]
assert substep_path[:11] == list(range(998, 987, -1))
assert substep_path[-1] == 0
print("Teacher evaluations per image:", len(substep_path) - 1)


@torch.no_grad()
def direct_teacher_sample(noise, path):
    x = noise.to(device)

    for high, low in zip(path[:-1], path[1:]):
        t = torch.full((len(x),), high, dtype=torch.long, device=device)
        eps = teacher(x, t)
        a_high = alpha_time_t[high]
        a_low = alpha_time_t[low]

        x0_pred = (x - (1 - a_high).sqrt() * eps) / a_high.sqrt()
        x = a_low.sqrt() * x0_pred + (1 - a_low).sqrt() * eps

        if not torch.isfinite(x).all():
            raise FloatingPointError(f"Non-finite teacher output at {high} -> {low}")

    return x.cpu()


def make_noise(count, seed):
    generator = torch.Generator(device="cpu").manual_seed(seed)
    return torch.randn(count, 3, 32, 32, generator=generator)

In [ ]:
TRAIN_PAIRS = 2048
TRAIN_SEED = 23456
BATCH_SIZE = 32
TRAIN_STEPS = 12000
TEST_IMAGES = 5000
TEST_SEED = 89012
if TEST_SEED == TRAIN_SEED:
    raise ValueError("Training and test noise seeds must differ.")

run_dir = Path(tempfile.mkdtemp(
    prefix="direct_teacher_target_final_", dir=TEACHER_PATH.parent
))
print("New output folder:", run_dir)

train_noise = make_noise(TRAIN_PAIRS, TRAIN_SEED)
target_batches = []
print(f"Generating {TRAIN_PAIRS} teacher-target training pairs...")
for start in range(0, TRAIN_PAIRS, BATCH_SIZE):
    batch = train_noise[start:start + BATCH_SIZE]
    target_batches.append(direct_teacher_sample(batch, substep_path))
    done = start + len(batch)
    if done % 256 == 0 or done == TRAIN_PAIRS:
        print(f"  {done}/{TRAIN_PAIRS}")

train_targets = torch.cat(target_batches)
if train_targets.shape != train_noise.shape or not torch.isfinite(train_targets).all():
    raise FloatingPointError("Teacher training targets have an invalid shape or contain non-finite values.")

train_pairs_path = run_dir / "train_pairs.pt"
with open(train_pairs_path, "xb") as f:
    torch.save({
        "noise": train_noise,
        "targets": train_targets,
        "seed": TRAIN_SEED,
        "teacher_sha256": EXPECTED_TEACHER_SHA256,
        "teacher_path": substep_path,
    }, f)
print("Saved training pairs:", train_pairs_path)

In [ ]:
print(f"Training for {TRAIN_STEPS} updates...")
training_start = time.time()
window_loss = 0.0
for step in range(1, TRAIN_STEPS + 1):
    indices = torch.randint(len(train_noise), (BATCH_SIZE,), generator=batch_rng)
    x = train_noise[indices].to(device)
    target = train_targets[indices].to(device)
    t = torch.full((len(x),), 998, dtype=torch.long, device=device)

    prediction = student(x, t)
    loss = F.mse_loss(prediction, target)
    if not torch.isfinite(loss):
        raise FloatingPointError(f"Non-finite training loss at step {step}")

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0, error_if_nonfinite=True)
    optimizer.step()

    window_loss += loss.item()
    if step % 500 == 0 or step == TRAIN_STEPS:
        window_size = 500 if step % 500 == 0 else step % 500
        print(f"  step {step}/{TRAIN_STEPS}: mean batch MSE={window_loss / window_size:.6f}")
        window_loss = 0.0

training_seconds = time.time() - training_start
model_config = {
    "method": "direct teacher-output regression",
    "teacher_checkpoint": str(TEACHER_PATH),
    "alpha_time_t": alpha_time_t.cpu(),
    "teacher_path": substep_path,
    "sampling_timestep": 998,
    "training_pairs": TRAIN_PAIRS,
    "training_noise_seed": TRAIN_SEED,
    "batch_seed": 56789,
    "batch_size": BATCH_SIZE,
    "learning_rate": 1e-4,
    "gradient_clip_norm": 1.0,
    "training_steps": TRAIN_STEPS,
    "test_noise_seed": TEST_SEED,
    "test_images": TEST_IMAGES,
    "sigma_data": student.sigma_data,
}
checkpoint_path = run_dir / "final_student.pt"
with open(checkpoint_path, "xb") as f:
    torch.save({
        "step": TRAIN_STEPS,
        "student": student.state_dict(),
        "optimizer": optimizer.state_dict(),
        "batch_generator_state": batch_rng.get_state(),
        "config": model_config,
        "training_seconds": training_seconds,
    }, f)
print(f"Training finished in {training_seconds / 60:.1f} minutes.")
print("Saved student and optimizer:", checkpoint_path)

In [ ]:
fid_root = Path(tempfile.mkdtemp(prefix="direct_teacher_fid_", dir="/content"))
real_dir = fid_root / "real"
student_dir = fid_root / "student"
teacher_dir = fid_root / "teacher"
for folder in (real_dir, student_dir, teacher_dir):
    folder.mkdir()

real_dataset = torchvision.datasets.CIFAR10(
    root="/content/data", train=True, download=True
)
if len(real_dataset) < TEST_IMAGES:
    raise ValueError("The CIFAR-10 reference split has fewer images than requested.")

print(f"Saving {TEST_IMAGES} real CIFAR-10 training images...")
for i in range(TEST_IMAGES):
    image, _ = real_dataset[i]
    image.save(real_dir / f"{i:05d}.png")

test_rng = torch.Generator(device="cpu").manual_seed(TEST_SEED)
student.eval()
student_below = 0
student_above = 0
teacher_below = 0
teacher_above = 0
pixel_count = 0
squared_error_sum = 0.0
print(f"Generating {TEST_IMAGES} matched student and teacher images...")
with torch.inference_mode():
    for start in range(0, TEST_IMAGES, BATCH_SIZE):
        count = min(BATCH_SIZE, TEST_IMAGES - start)
        noise = torch.randn(count, 3, 32, 32, generator=test_rng)
        t = torch.full((count,), 998, dtype=torch.long, device=device)
        student_output = student(noise.to(device), t).cpu()
        teacher_output = direct_teacher_sample(noise, substep_path)
        if student_output.shape != noise.shape or teacher_output.shape != noise.shape:
            raise ValueError(f"Invalid output shape at test image {start}")
        if not torch.isfinite(student_output).all() or not torch.isfinite(teacher_output).all():
            raise FloatingPointError(f"Non-finite output at test image {start}")

        student_below += (student_output < 0).sum().item()
        student_above += (student_output > 1).sum().item()
        teacher_below += (teacher_output < 0).sum().item()
        teacher_above += (teacher_output > 1).sum().item()
        pixel_count += student_output.numel()
        squared_error_sum += (student_output - teacher_output).square().sum().item()

        student_pixels = student_output.clamp(0, 1).mul(255).round().to(torch.uint8)
        teacher_pixels = teacher_output.clamp(0, 1).mul(255).round().to(torch.uint8)
        for j in range(count):
            name = f"{start + j:05d}.png"
            Image.fromarray(student_pixels[j].permute(1, 2, 0).numpy()).save(student_dir / name)
            Image.fromarray(teacher_pixels[j].permute(1, 2, 0).numpy()).save(teacher_dir / name)

        done = start + count
        if done % 1024 == 0 or done == TEST_IMAGES:
            print(f"  {done}/{TEST_IMAGES}")

for folder in (real_dir, student_dir, teacher_dir):
    if len(list(folder.glob("*.png"))) != TEST_IMAGES:
        raise RuntimeError(f"Incorrect number of images in {folder}")
print("FID images prepared:", fid_root)

In [ ]:
print("Calculating student FID...")
student_fid = calculate_fid_given_paths(
    [str(real_dir), str(student_dir)], batch_size=50, device=device, dims=2048
)
print("Calculating teacher FID...")
teacher_fid = calculate_fid_given_paths(
    [str(real_dir), str(teacher_dir)], batch_size=50, device=device, dims=2048
)
if not math.isfinite(student_fid) or not math.isfinite(teacher_fid):
    raise FloatingPointError("A calculated FID is not finite.")

digest = hashlib.sha256()
with open(checkpoint_path, "rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        digest.update(chunk)

results = {
    "method": "direct teacher-output regression",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "student_fid": float(student_fid),
    "teacher_fid": float(teacher_fid),
    "paired_raw_mse": squared_error_sum / pixel_count,
    "test_images": TEST_IMAGES,
    "test_noise_seed": TEST_SEED,
    "real_reference": f"first {TEST_IMAGES} CIFAR-10 training images",
    "real_preprocessing": "original CIFAR-10 uint8 PNG",
    "generated_preprocessing": "clamp [0,1], round to uint8 PNG",
    "fid_implementation": "pytorch-fid 0.3.0, Inception features 2048",
    "student_evaluations_per_image": 1,
    "teacher_evaluations_per_image": len(substep_path) - 1,
    "student_fraction_below_zero": student_below / pixel_count,
    "student_fraction_above_one": student_above / pixel_count,
    "teacher_fraction_below_zero": teacher_below / pixel_count,
    "teacher_fraction_above_one": teacher_above / pixel_count,
    "student_checkpoint": str(checkpoint_path),
    "student_checkpoint_sha256": digest.hexdigest(),
    "training_pairs": TRAIN_PAIRS,
    "training_noise_seed": TRAIN_SEED,
    "training_steps": TRAIN_STEPS,
    "training_batch_size": BATCH_SIZE,
    "learning_rate": 1e-4,
    "training_seconds": training_seconds,
    "torch_version": str(torch.__version__),
    "torchvision_version": str(torchvision.__version__),
    "gpu": torch.cuda.get_device_name(0),
}
result_path = run_dir / "fid_results.json"
with open(result_path, "x") as f:
    json.dump(results, f, indent=2, allow_nan=False)

print(f"\nStudent FID ({TEST_IMAGES} images): {student_fid:.3f}")
print(f"Teacher FID ({TEST_IMAGES} images): {teacher_fid:.3f}")
print(f"Matched raw pixel MSE: {results['paired_raw_mse']:.6f}")
print(f"Student below/above [0,1]: {student_below / pixel_count:.2%} / {student_above / pixel_count:.2%}")
print(f"Teacher below/above [0,1]: {teacher_below / pixel_count:.2%} / {teacher_above / pixel_count:.2%}")
print("Saved checkpoint:", checkpoint_path)
print("Saved FID results:", result_path)